# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duaf9877/FlyRank-AI-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

The paper reports a decline-detection result based on an outcome label. The methodology question I would ask is: Where exactly does the label come from, and is it directly observed or a proxy created from other measurements?

This matters because model performance can only be interpreted relative to the quality of the label. If the label is a proxy, a strong model score may show that the model predicts the proxy well, but not necessarily the broader business outcome perfectly.

For my own work, I would want the same question asked. My Week 5 target is:

target = 1 if trend_direction == "down"

This means the label is derived from the dataset's trend_direction column rather than being a separate independently collected outcome. I therefore treat the model as predicting this defined decline proxy, not proving that a page is objectively declining in every possible sense.

he paper reports performance findings that suggest the model can generalize beyond the training data. The methodology question I would ask is: Does the validation design match the way the model would actually be used?

A random row-level split can make performance appear stronger if related rows from the same client, user, site, or time period appear in both training and testing data. In that situation, the model may partially learn patterns that are already represented in the training data.

A grouped or time-aware split provides a stronger test when repeated entities or time patterns are present.

For my own model, I apply this same question by comparing a random split with a client_id-grouped split. The grouped evaluation asks a harder question:

How well does the model perform on clients it did not see during training?

Therefore, I interpret the grouped result as the more relevant estimate for generalization to unseen clients. A difference between the random and grouped scores is itself a useful finding rather than something to hide.

The dataset guidance explicitly warns that client_id should be used for grouped train/test splits and that IDs should not be used as model features.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/duaf9877/FlyRank-AI-ML-Internship/main/data/raw/content_refresh_anonymized.csv"
)

print("Dataset shape:", df.shape)
print("\nNumber of clients:", df["client_id"].nunique())

Dataset shape: (30000, 44)

Number of clients: 32


In [3]:
# The target is derived from trend_direction.
# trend_direction itself must NOT be used as a feature.

df["target"] = (df["trend_direction"] == "down").astype(int)

print("Declining pages:", df["target"].sum())
print("Base rate:", round(df["target"].mean(), 4))
print("Base rate percentage:", round(df["target"].mean() * 100, 2), "%")

Declining pages: 16262
Base rate: 0.5421
Base rate percentage: 54.21 %


#Markdown cell
Base rate

The positive class rate is printed before evaluating the model because accuracy must be interpreted relative to a simple baseline. If the model score is only slightly above the majority-class baseline, the apparent performance may not represent much predictive value.

In [4]:
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "pageviews_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "ctr",
    "engagement_rate",
    "content_type",
    "main_intent",
    "provider_used"
]

X = df[features]
y = df["target"]
groups = df["client_id"]

numeric = X.select_dtypes(include=np.number).columns.tolist()
categorical = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric features:", numeric)
print("Categorical features:", categorical)

Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'pageviews_90d', 'users_90d', 'engaged_sessions_90d', 'days_since_last_update', 'content_age_days', 'avg_position', 'ctr', 'engagement_rate']
Categorical features: ['content_type', 'main_intent', 'provider_used']


In [5]:
def make_model():

    preprocess = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numeric
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]),
                categorical
            )
        ]
    )

    return Pipeline([
        ("prep", preprocess),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

In [6]:
def evaluate_model(model, X_train, X_test, y_train, y_test, split_name):

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    results = {
        "Split": split_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0),
        "Test Positive Rate": y_test.mean()
    }

    return results, predictions

In [7]:
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = make_model()

random_results, random_predictions = evaluate_model(
    random_model,
    X_train_random,
    X_test_random,
    y_train_random,
    y_test_random,
    "Random row-level split"
)

random_results

{'Split': 'Random row-level split',
 'Accuracy': 0.6131666666666666,
 'Precision': 0.6146269391775425,
 'Recall': 0.7675276752767528,
 'F1 Score': 0.6826199917954328,
 'Test Positive Rate': np.float64(0.542)}

In [8]:
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_splitter.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

group_model = make_model()

group_results, group_predictions = evaluate_model(
    group_model,
    X_train_group,
    X_test_group,
    y_train_group,
    y_test_group,
    "Grouped by client"
)

group_results

{'Split': 'Grouped by client',
 'Accuracy': 0.5375628752231056,
 'Precision': 0.5401126911725248,
 'Recall': 0.6392505557319784,
 'F1 Score': 0.5855148342059336,
 'Test Positive Rate': np.float64(0.5109524582184002)}

In [9]:
comparison = pd.DataFrame([
    random_results,
    group_results
])

comparison

,Split,Accuracy,Precision,Recall,F1 Score,Test Positive Rate
0,Random row-level split,0.613167,0.614627,0.767528,0.682620,0.542000
1,Grouped by client,0.537563,0.540113,0.639251,0.585515,0.510952


In [10]:
train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

overlap = train_clients.intersection(test_clients)

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Overlapping clients:", len(overlap))

assert len(overlap) == 0, "Client leakage detected!"

Training clients: 25
Testing clients: 7
Overlapping clients: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

#Leakage audit approach

I checked the final feature set for three main types of leakage:

Label-derived leakage — columns used to create the target or closely derived from it.
Future or overlapping information — features that would not be available at prediction time.
Decision-derived leakage — product scores or recommendation flags that may encode an existing decision.

The target is derived from trend_direction, so trend_direction and trend_pct are excluded from the model. The data guidance explicitly identifies this as a label trap. Product flags are also treated as leakage-risk features rather than model inputs.

In [11]:
suspect_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining",
    "is_quick_win",
    "needs_indexing",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "ai_opportunity",
    "is_underperformer",
    "is_initial_refresh_candidate",
    "health_score"
]

leakage_audit = pd.DataFrame({
    "Column": suspect_columns,
    "Present in Dataset": [
        column in df.columns for column in suspect_columns
    ],
    "Used as Model Feature": [
        column in features for column in suspect_columns
    ]
})

leakage_audit

,Column,Present in Dataset,Used as Model Feature
0,trend_direction,True,False
1,trend_pct,True,False
2,is_declining,False,False
3,is_quick_win,False,False
4,needs_indexing,False,False
5,needs_ctr_fix,False,False
6,needs_engagement_fix,False,False
7,ai_opportunity,False,False
8,is_underperformer,False,False
9,is_initial_refresh_candidate,False,False


In [12]:
id_columns = ["content_id", "client_id"]

for column in id_columns:
    print(
        f"{column}:",
        "USED AS FEATURE" if column in features else "NOT USED AS FEATURE"
    )

content_id: NOT USED AS FEATURE
client_id: NOT USED AS FEATURE


In [13]:
for column in ["trend_direction", "trend_pct", "is_declining"]:
    print(
        f"{column}:",
        "LEAKAGE RISK - incorrectly included"
        if column in features
        else "Excluded"
    )

trend_direction: Excluded
trend_pct: Excluded
is_declining: Excluded


In [14]:
# LEGAL model: original feature set
honest_features = features.copy()

# DELIBERATELY LEAKY model: adds trend_pct,
# which is directly related to the trend-based label.
leaky_features = features + ["trend_pct"]

X_honest = df[honest_features]
X_leaky = df[leaky_features]

# Same grouped split
X_train_honest = X_honest.iloc[train_idx]
X_test_honest = X_honest.iloc[test_idx]

X_train_leaky = X_leaky.iloc[train_idx]
X_test_leaky = X_leaky.iloc[test_idx]

In [15]:
# Honest model score
honest_model = make_model()

honest_model.fit(
    X_train_honest,
    y_train_group
)

honest_predictions = honest_model.predict(X_test_honest)

honest_f1 = f1_score(
    y_test_group,
    honest_predictions,
    zero_division=0
)

print("Honest grouped F1:", round(honest_f1, 4))

Honest grouped F1: 0.5855


In [16]:
leaky_numeric = X_leaky.select_dtypes(
    include=np.number
).columns.tolist()

leaky_categorical = X_leaky.select_dtypes(
    exclude=np.number
).columns.tolist()

leaky_preprocess = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        leaky_numeric
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        leaky_categorical
    )
])

leaky_model = Pipeline([
    ("prep", leaky_preprocess),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

leaky_model.fit(
    X_train_leaky,
    y_train_group
)

leaky_predictions = leaky_model.predict(X_test_leaky)

leaky_f1 = f1_score(
    y_test_group,
    leaky_predictions,
    zero_division=0
)

print("Leaky grouped F1:", round(leaky_f1, 4))

Leaky grouped F1: 0.8502


In [17]:
leakage_test = pd.DataFrame({
    "Model": [
        "Honest feature set",
        "Feature set with deliberate leakage"
    ],
    "F1 Score": [
        honest_f1,
        leaky_f1
    ]
})

leakage_test

,Model,F1 Score
0,Honest feature set,0.585515
1,Feature set with deliberate leakage,0.850195


#Leakage audit verdict

Verdict: the final model feature set is designed to avoid the identified label-derived and decision-derived leakage risks.

trend_direction is excluded because it directly defines the target.
trend_pct is excluded because the data documentation identifies it as related to the target definition.
Product decision flags and scores are excluded.
content_id and client_id are not used as model features.
client_id is used only for grouped validation.

The deliberate leakage test is included to demonstrate how a suspicious feature can make evaluation appear stronger than a legally available feature set.

The honest model result should be used for reporting and decision-support. The deliberately leaky result is included only as a validation check and must not be treated as deployable performance.

In [18]:
failure_examples = df.iloc[test_idx].copy()

failure_examples["actual"] = y_test_group.values
failure_examples["predicted"] = group_predictions

failure_examples["prediction_type"] = np.where(
    (failure_examples["actual"] == 1) &
    (failure_examples["predicted"] == 0),
    "False Negative",
    np.where(
        (failure_examples["actual"] == 0) &
        (failure_examples["predicted"] == 1),
        "False Positive",
        "Correct"
    )
)

errors = failure_examples[
    failure_examples["prediction_type"] != "Correct"
]

errors["prediction_type"].value_counts()

,count
prediction_type,
False Positive,1714
False Negative,1136


In [19]:
false_positives = errors[
    errors["prediction_type"] == "False Positive"
][
    [
        "content_id",
        "client_id",
        "content_type",
        "main_intent",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "actual",
        "predicted"
    ]
].head(5)

false_positives

,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,actual,predicted
13,content_a5a2fbc76336,client_8527a891e2,keyword article,informational,307,0,0.00,39.8,103,0,1
26,content_72c5c2d73e5a,client_4e07408562,keyword article,informational,2426,3,0.12,30.0,13,0,1
36,content_bce275871a25,client_f369cb89fc,keyword article,informational,371,5,1.35,5.4,20,0,1
56,content_dcebfd222b10,client_f369cb89fc,keyword article,informational,16,0,0.00,4.6,20,0,1
64,content_685de0e3b7cb,client_f369cb89fc,keyword article,informational,2639,3,0.11,7.2,8,0,1


In [20]:
false_negatives = errors[
    errors["prediction_type"] == "False Negative"
][
    [
        "content_id",
        "client_id",
        "content_type",
        "main_intent",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "actual",
        "predicted"
    ]
].head(5)

false_negatives

,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,ctr,avg_position,days_since_last_update,actual,predicted
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,0.05,20.3,25,1,0
23,content_2da6ae9d0882,client_e629fa6598,keyword article,informational,297,1,0.34,13.9,20,1,0
54,content_ff8ea1364b59,client_e629fa6598,keyword article,informational,170,0,0.00,10.7,22,1,0
58,content_caff51984338,client_e629fa6598,keyword article,transactional,71,0,0.00,9.1,20,1,0
81,content_16788821b64a,client_e629fa6598,keyword article,informational,320,1,0.31,9.6,22,1,0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In this dataset, the Logistic Regression model measured useful classification performance under a client-grouped evaluation. The observed results suggest that the selected features contain directional information related to the dataset's decline label. The model should therefore be treated as a decision-support signal rather than proof that a page will decline or that the model will generalize to every future client or environment.\

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.